# DataHandler Method Comparison

**Goal**: Compare the OLD vs NEW method for loading WFS data.

- **Method A (OLD)**: `create_data_from_remote()` - Fetches WFS data live (SLOW)
- **Method B (NEW)**: `load_remote_data_from_geopackage()` - Loads from file (FAST)

**Expected Result**: Both methods produce identical output!

## Setup

In [1]:
import sys
sys.path.append("../")

import src.paths as PATHS
import src.constants as CONST
import src.data.data_handler as DH
import src.data.config as DATA_CONFIG

import geopandas as gpd
import pandas as pd
from pathlib import Path
import time

print("✅ Setup complete!")

✅ Setup complete!


## Load Input Data

We'll use a SMALL subset (3 regions) to make Method A fast enough for testing.

In [2]:
# Load prediction regions from ORIGINAL geopackage (no WFS layers yet)
# Method A will fetch WFS data live, Method B will load from bundled file
original_gpkg = PATHS.DATA_DIR / "phase1_2025-08-14_v1.gpkg"
bundled_gpkg = PATHS.DATA_DIR / "phase1_2025-08-14_v1_w_wfs.gpkg"  # To be created by WFSDataBundler

prediction_regions_full = gpd.read_file(original_gpkg, layer="vlakken_scope")
river_centerline = gpd.read_file(original_gpkg, layer="middenlijn")

# Take ONLY 3 regions for testing (to make Method A reasonably fast)
prediction_regions = prediction_regions_full.head(10).copy()

local_geospatial_data = {
    CONST.AggregationOperations.CENTERLINE_SHAPE.value: river_centerline
}

print(f"📍 Testing with {len(prediction_regions)} regions (subset for speed)")
print(f"📐 CRS: {prediction_regions.crs}")
print(f"\n🗺️ Region IDs: {prediction_regions['location_id'].tolist()}")

📍 Testing with 10 regions (subset for speed)
📐 CRS: EPSG:28992

🗺️ Region IDs: ['maas_l_2180_2181', 'maas_l_2181_2182', 'maas_l_2182_2183', 'maas_l_2183_2184', 'maas_l_2184_2185', 'maas_l_2185_2186', 'maas_l_2186_2187', 'maas_l_2187_2188', 'maas_l_2188_2189', 'maas_l_2189_2190']


In [3]:
# Create configuration
config = DATA_CONFIG.DataConfiguration()

print(f"✅ Configuration created")
print(f"   - WFS services: {len(config.known_wfs_services)}")
print(f"   - Feature layers: {len(config.feature_creation_config)}")
print(f"   - Buffer: {config.prediction_region_buffer}m")

✅ Configuration created
   - WFS services: 3
   - Feature layers: 5
   - Buffer: 10m


---

## Method A: create_data_from_remote() - OLD METHOD

Fetches WFS data live for each region. **SLOW** (even for 3 regions).

In [4]:
print("\n" + "="*60)
print("METHOD A: create_data_from_remote() - Fetch Live Data")
print("="*60)

# Create DataHandler instance
data_handler_a = DH.DataHandler(
    config=config,
    prediction_regions=prediction_regions,
    local_data_for_enrichment=local_geospatial_data,
)

# Time the operation
start_time = time.time()
data_handler_a.create_data_from_remote()
elapsed_a = time.time() - start_time

print(f"\n⏱️  Method A completed in {elapsed_a:.2f} seconds")
print(f"📊 Output shape: {data_handler_a.scope_region_features.shape}")

Setting the number extra features to 0, even though it should be automatically calculated.
/Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



METHOD A: create_data_from_remote() - Fetch Live Data


Fetching WFS data:   0%|          | 0/10 [00:00<?, ?region/s]The geodataframe is empty and thus the area fraction cannot be calculated. It is assumed it would be 0.
The geodataframe is empty and the majority class cannot be calculated for ['gebruiksdoel'].
No feature configuration found for rws_vegetatielegger:heggen of the WFS vegetation, skipping.Why are we downloading the data though?
Fetching WFS data:  10%|█         | 1/10 [00:01<00:13,  1.54s/region]The geodataframe is empty and thus the area fraction cannot be calculated. It is assumed it would be 0.
The geodataframe is empty and the majority class cannot be calculated for ['gebruiksdoel'].
No feature configuration found for rws_vegetatielegger:heggen of the WFS vegetation, skipping.Why are we downloading the data though?
Fetching WFS data:  20%|██        | 2/10 [00:03<00:12,  1.53s/region]Multiple majority classes found in some of the columns, we only take the first one.
The geodataframe is empty and thus the area fraction cann


⏱️  Method A completed in 16.12 seconds
📊 Output shape: (10, 11)


---

## Method B: load_remote_data_from_geopackage() - NEW METHOD

Loads pre-fetched WFS data from geopackage. **FAST**!

In [5]:
print("\n" + "="*60)
print("METHOD B: load_remote_data_from_geopackage() - Load From File")
print("="*60)

# Check if bundled geopackage exists
if not bundled_gpkg.exists():
    print(f"\n⚠️  BUNDLED GEOPACKAGE NOT FOUND: {bundled_gpkg.name}")
    print(f"\n📝 You need to create it first using WFSDataBundler:")
    print(f"   1. Open test_wfs_bundler.ipynb")
    print(f"   2. Run it to create the bundled geopackage with new naming convention")
    print(f"   3. Come back here to test Method B")
    print(f"\n⏭️  Skipping Method B for now...")
    data_handler_b = None
    elapsed_b = None
else:
    # Create DataHandler instance
    data_handler_b = DH.DataHandler(
        config=config,
        prediction_regions=prediction_regions,
        local_data_for_enrichment=local_geospatial_data,
    )
    
    # Time the operation
    start_time = time.time()
    data_handler_b.load_remote_data_from_geopackage(
        gpkg_path=bundled_gpkg,
        show_progress=True,
    )
    elapsed_b = time.time() - start_time
    
    print(f"\n⏱️  Method B completed in {elapsed_b:.2f} seconds")
    print(f"📊 Output shape: {data_handler_b.scope_region_features.shape}")

Setting the number extra features to 0, even though it should be automatically calculated.



METHOD B: load_remote_data_from_geopackage() - Load From File


Processing regions:   0%|          | 0/10 [00:00<?, ?region/s]The geodataframe is empty and thus the area fraction cannot be calculated. It is assumed it would be 0.
The geodataframe is empty and the majority class cannot be calculated for ['gebruiksdoel'].
The geodataframe is empty and thus the area fraction cannot be calculated. It is assumed it would be 0.
The geodataframe is empty and the majority class cannot be calculated for ['gebruiksdoel'].
Multiple majority classes found in some of the columns, we only take the first one.
The geodataframe is empty and thus the area fraction cannot be calculated. It is assumed it would be 0.
The geodataframe is empty and the majority class cannot be calculated for ['gebruiksdoel'].
The geodataframe is empty and thus the density cannot be calculated. It is assumed it would be 0.
Multiple majority classes found in some of the columns, we only take the first one.
The geodataframe is empty and thus the area fraction cannot be calculated. It is ass


⏱️  Method B completed in 0.08 seconds
📊 Output shape: (10, 11)


---

## 🔍 COMPARISON

Let's compare the outputs to verify they're identical!

### 1. Shape Comparison

In [12]:
if data_handler_b is None:
    print("\n⚠️  Skipping comparison - Method B was not run (bundled geopackage not found)")
else:
    print("\n" + "="*60)
    print("SHAPE COMPARISON")
    print("="*60)
    
    shape_a = data_handler_a.scope_region_features.shape
    shape_b = data_handler_b.scope_region_features.shape
    
    print(f"\nMethod A (old): {shape_a}")
    print(f"Method B (new): {shape_b}")
    
    if shape_a == shape_b:
        print("\n✅ SHAPES MATCH!")
    else:
        print("\n❌ SHAPES DO NOT MATCH!")
        print(f"   Difference: {shape_a[0] - shape_b[0]} rows, {shape_a[1] - shape_b[1]} columns")


SHAPE COMPARISON

Method A (old): (10, 11)
Method B (new): (10, 11)

✅ SHAPES MATCH!


### 2. Column Comparison

In [13]:
if data_handler_b is None:
    print("\n⚠️  Skipping comparison - Method B was not run")
else:
    print("\n" + "="*60)
    print("COLUMN COMPARISON")
    print("="*60)
    
    cols_a = set(data_handler_a.scope_region_features.columns)
    cols_b = set(data_handler_b.scope_region_features.columns)
    
    print(f"\nMethod A: {len(cols_a)} columns")
    print(f"Method B: {len(cols_b)} columns")
    
    # Check for differences
    only_in_a = cols_a - cols_b
    only_in_b = cols_b - cols_a
    
    if cols_a == cols_b:
        print("\n✅ COLUMNS MATCH!")
        print("\nColumn names:")
        for col in sorted(cols_a):
            print(f"  - {col}")
    else:
        print("\n❌ COLUMNS DO NOT MATCH!")
        if only_in_a:
            print(f"\nOnly in Method A: {only_in_a}")
        if only_in_b:
            print(f"\nOnly in Method B: {only_in_b}")


COLUMN COMPARISON

Method A: 11 columns
Method B: 11 columns

✅ COLUMNS MATCH!

Column names:
  - BrpGewas_area_fraction
  - BrpGewas_majority_class_category
  - BrpGewas_majority_class_gewas
  - bag:pand_area_fraction
  - bag:pand_majority_class_gebruiksdoel
  - end_year
  - geometry
  - location_id
  - rws_vegetatielegger:bomen_numerical_density
  - rws_vegetatielegger:vegetatieklassen_majority_class_vlklasse
  - start_year


### 3. Data Type Comparison

In [14]:
if data_handler_b is None:
    print("\n⚠️  Skipping comparison - Method B was not run")
else:
    print("\n" + "="*60)
    print("DATA TYPE COMPARISON")
    print("="*60)
    
    dtypes_a = data_handler_a.scope_region_features.dtypes
    dtypes_b = data_handler_b.scope_region_features.dtypes
    
    # Compare dtypes for matching columns
    matching_cols = cols_a & cols_b
    dtype_mismatches = []
    
    for col in sorted(matching_cols):
        if dtypes_a[col] != dtypes_b[col]:
            dtype_mismatches.append((col, dtypes_a[col], dtypes_b[col]))
    
    if not dtype_mismatches:
        print("\n✅ DATA TYPES MATCH!")
    else:
        print("\n❌ DATA TYPE MISMATCHES FOUND:")
        for col, dtype_a, dtype_b in dtype_mismatches:
            print(f"  {col}: {dtype_a} vs {dtype_b}")


DATA TYPE COMPARISON

✅ DATA TYPES MATCH!


### 4. Sample Values Comparison

In [15]:
print("\n" + "="*60)
print("METHOD A - FIRST 3 ROWS")
print("="*60)

data_handler_a.scope_region_features.head(10)


METHOD A - FIRST 3 ROWS


,location_id,start_year,end_year,geometry,BrpGewas_area_fraction,BrpGewas_majority_class_category,BrpGewas_majority_class_gewas,bag:pand_area_fraction,bag:pand_majority_class_gebruiksdoel,rws_vegetatielegger:bomen_numerical_density,rws_vegetatielegger:vegetatieklassen_majority_class_vlklasse
0,maas_l_2180_2181,2016,2024,"POLYGON ((148604.881 416309.328, 148538.207 41...",0.156551,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",0.0,None,0.000058,Bos
1,maas_l_2181_2182,2016,2024,"POLYGON ((148491.678 416329.661, 148416.977 41...",0.370831,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",0.0,None,0.000056,Water
2,maas_l_2182_2183,2016,2024,"POLYGON ((148373.399 416364.687, 148290.662 41...",0.146087,Grasland,"Grasland, blijvend",0.0,None,0.000000,Gras en Akker
3,maas_l_2183_2184,2016,2024,"POLYGON ((148268.941 416410.637, 148182.141 41...",0.327541,Grasland,"Grasland, blijvend",0.0,None,0.000063,Riet en Ruigte
4,maas_l_2184_2185,2016,2024,"POLYGON ((148182.141 416457.702, 148095.34 416...",0.254791,Grasland,"Grasland, blijvend",0.0,None,0.000127,Riet en Ruigte
5,maas_l_2185_2186,2016,2024,"POLYGON ((148095.34 416504.766, 148012.778 416...",0.131474,Grasland,"Grasland, blijvend",0.0,None,0.000065,Riet en Ruigte
6,maas_l_2186_2187,2016,2024,"POLYGON ((148012.778 416549.074, 147924.54 416...",0.061409,Grasland,"Grasland, blijvend",0.0,None,0.000063,Struweel
7,maas_l_2187_2188,2016,2024,"POLYGON ((147924.54 416593.386, 147847.709 416...",0.194674,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",0.0,None,0.000000,Riet en Ruigte
8,maas_l_2188_2189,2016,2024,"POLYGON ((147847.709 416622.478, 147768.88 416...",0.330553,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",0.0,None,0.000138,Struweel
9,maas_l_2189_2190,2016,2024,"POLYGON ((147768.88 416644.29, 147685.898 4166...",0.303739,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",0.0,None,0.000000,Bos


In [ ]:
print("\n" + "="*60)
print("METHOD B - FIRST 3 ROWS")
print("="*60)

data_handler_b.scope_region_features.head(10)


METHOD B - FIRST 3 ROWS


,location_id,start_year,end_year,geometry,BrpGewas_area_fraction,BrpGewas_majority_class_category,BrpGewas_majority_class_gewas,bag:pand_area_fraction,bag:pand_majority_class_gebruiksdoel,rws_vegetatielegger:bomen_numerical_density,rws_vegetatielegger:vegetatieklassen_majority_class_vlklasse
0,maas_l_2180_2181,2016,2024,"POLYGON ((148604.881 416309.328, 148538.207 41...",0.156551,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",0.0,None,0.000058,Bos
1,maas_l_2181_2182,2016,2024,"POLYGON ((148491.678 416329.661, 148416.977 41...",0.370831,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",0.0,None,0.000056,Water
2,maas_l_2182_2183,2016,2024,"POLYGON ((148373.399 416364.687, 148290.662 41...",0.146087,Grasland,"Grasland, blijvend",0.0,None,0.000000,Gras en Akker
3,maas_l_2183_2184,2016,2024,"POLYGON ((148268.941 416410.637, 148182.141 41...",0.327541,Grasland,"Grasland, blijvend",0.0,None,0.000063,Riet en Ruigte
4,maas_l_2184_2185,2016,2024,"POLYGON ((148182.141 416457.702, 148095.34 416...",0.254791,Grasland,"Grasland, blijvend",0.0,None,0.000127,Riet en Ruigte
5,maas_l_2185_2186,2016,2024,"POLYGON ((148095.34 416504.766, 148012.778 416...",0.131474,Grasland,"Grasland, blijvend",0.0,None,0.000065,Riet en Ruigte
6,maas_l_2186_2187,2016,2024,"POLYGON ((148012.778 416549.074, 147924.54 416...",0.061409,Grasland,"Grasland, blijvend",0.0,None,0.000063,Struweel
7,maas_l_2187_2188,2016,2024,"POLYGON ((147924.54 416593.386, 147847.709 416...",0.194674,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",0.0,None,0.000000,Riet en Ruigte
8,maas_l_2188_2189,2016,2024,"POLYGON ((147847.709 416622.478, 147768.88 416...",0.330553,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",0.0,None,0.000138,Struweel
9,maas_l_2189_2190,2016,2024,"POLYGON ((147768.88 416644.29, 147685.898 4166...",0.303739,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",0.0,None,0.000000,Bos


### 5. Performance Comparison

In [11]:
if data_handler_b is None:
    print("\n⚠️  Skipping comparison - Method B was not run")
else:
    print("\n" + "="*60)
    print("PERFORMANCE COMPARISON")
    print("="*60)
    
    print(f"\nMethod A (old): {elapsed_a:.2f} seconds")
    print(f"Method B (new): {elapsed_b:.2f} seconds")
    
    if elapsed_b < elapsed_a:
        speedup = elapsed_a / elapsed_b
        print(f"\n✅ Method B is {speedup:.1f}x FASTER!")
    else:
        print(f"\n⚠️  Method B took longer (unexpected!)")


PERFORMANCE COMPARISON

Method A (old): 16.12 seconds
Method B (new): 0.08 seconds

✅ Method B is 192.0x FASTER!


---

## ✅ Summary

If all checks pass:
- ✅ Shapes match
- ✅ Columns match
- ✅ Data types match
- ✅ Method B is faster

Then the new method is **PRODUCTION READY**! 🎉